<table>
<tr>
<td style="vertical-align: middle; border: none;">

# Streaming NASA Earth Data with *earthaccess*

</td>
<td style="vertical-align: middle; border: none;">

<img src="https://earthdata.nasa.gov/s3fs-public/styles/hds_generic_card/public/2024-01/earthaccess%20logo%20good.png" width="120">
</td>
</tr>
</table>

<center><h3>From NASA Earthdata to your Python workflow — without downloading the file</h2></center>
<br>

In this notebook, we'll explore how the [*earthaccess*](https://github.com/nsidc/earthaccess) Python library can **stream NASA Earthdata directly into memory**, allowing you to work with remote data without first saving entire file to disk.

We'll use **MERRA-2**, a global NASA atmospheric reanalysis dataset, to demonstrate:

* **Download → Disk → Memory**  
vs.  
* **Stream → Memory**

---

### **Andrew Barrett<sup>1</sup> · Luis López<sup>1</sup>**  
*Nasa Earthdata Webinar · August 27, 2026*

<sup>1</sup> **University of Colorado Boulder, National Snow and Ice Data Center (NSIDC)**

## 1. Requirements

We need a recent version of `earthaccess` as the library changed a few times as the project matured, this notebook uses the current release (v0.18.0), where `earthaccess.login()` is required *before* we can search, download, or open/stream anything.

We'll also need an Earthdata Login (EDL) account. If you don't have one yet, create one for free at https://urs.earthdata.nasa.gov/users/new.


In [ ]:
# Uncomment to install/upgrade in this environment
# %pip install -U earthaccess xarray dask h5netcdf netCDF4 fsspec s3fs matplotlib hvplot

In [ ]:
import warnings

import earthaccess
import xarray as xr

warnings.filterwarnings("ignore")

# First we authenticate ourselves with NASA EDL credentials
auth = earthaccess.login()

print("Authenticated:", auth.authenticated)
print("earthaccess version:", earthaccess.__version__)

## 2. Downloading vs. Streaming — what's the actual difference?

Both paths start the same way: we search the Common Metadata Repository (CMR) for granules, `earthaccess` resolves the data URLs (increasingly these are direct S3 URLs in the Earthdata Cloud, not just HTTPS), and it opens an authenticated connection. Concretely, that means either exchanging our EDL bearer token for short-lived, temporary AWS credentials via NASA's S3 credential endpoint (for direct `s3://` access from in-region compute), or attaching the bearer token as an `Authorization` header on HTTPS requests (for out-of-region / non-AWS access, redirected through EDL's OAuth flow). `earthaccess` picks whichever path applies automatically based on where our code is running and what the granule's links support.

The difference is what happens *after* that connection is open:

| | `earthaccess.download()` | `earthaccess.open()` |
|---|---|---|
| **What happens** | <div style="padding: 12px 0;">Streams the *entire* file's bytes from S3/HTTPS to a local file on disk</div> | <div style="padding: 12px 0;">Returns file-like objects (`fsspec` file handles) that our reader library (e.g. `xarray`/`h5netcdf`) reads from lazily</div> |
| **Where the bytes land** | <div style="padding: 12px 0;">Local disk (or our compute environment's attached storage)</div> | <div style="padding: 12px 0;">RAM, and only the bytes actually requested</div> |
| **When bytes move** | <div style="padding: 12px 0;">All at once, up front, before we can touch the data</div> | <div style="padding: 12px 0;">On demand, in chunks, as our code accesses specific variables/slices</div> |
| **Disk footprint** | <div style="padding: 12px 0;">One full copy of every file we touch</div> | <div style="padding: 12px 0;">Minimal (until/unless we explicitly write something out)</div> |


The key mental shift: **streaming doesn't mean "no data movement."** It still moves bytes over the network — it just moves *only the bytes our analysis actually asks for*, and it does so lazily, which means not fetching all at once but only as the data is required (more on this in Section 5).

## 3. Search for MERRA-2 data

|  |  |
|---|---:|
| We'll use [**MERRA-2**](https://gmao.gsfc.nasa.gov/gmao-products/merra-2/) (Modern-Era Retrospective analysis for Research and Applications, Version 2), NASA's global atmospheric reanalysis product, produced by GMAO and distributed through GES DISC. <br><br>Specifically, [**M2T1NXSLV**](https://cmr.earthdata.nasa.gov/search/concepts/C1276812863-GES_DISC.html), an hourly, single-level diagnostics collection containing 2D meteorological fields such as 2-m air temperature, sea-level pressure, and 10-m winds. The data are provided on a global **0.625° × 0.5° longitude–latitude grid** and span from **1980 to the present**.<br><br>Data are distributed as **NetCDF-4 files**, with each daily file typically around **400 MB**. That means working with even a relatively short period can involve downloading many gigabytes of data, while the complete time series represents a much larger collection. *Note that we can use DOI to search for our files*| <img src="https://gmao.gsfc.nasa.gov/media/gmao_products/VPj9Vm07jIqPeIBhIWLGrweILf/atmosphere-surface-air-temp_m8s31t5requdycw2nblp6z4ox.png" width="130%"> |



In [ ]:
results = earthaccess.search_data(
    short_name="M2T1NXSLV",
    temporal=("2023-07-01", "2023-07-02"),
    count=2,
)

print(
    f"Total size for {len(results)} files: {round(sum([g.size() for g in results]), 2)} MB"
)

Each item in `results` is an `earthaccess.DataGranule` — it carries the metadata *and* the resolvable data links (both HTTPS and, where available, direct S3 URLs), without moving any of the actual science data yet. Searching CMR is metadata-only regardless of which path we take next.


## 4. Example A — the traditional *"download and open"* workflow

This is the workflow most people are used to from working with data on a local machine. When we use dataframe-like libraries like Xarray or Geopandas they usually query the files using a "driver" this is a layer in software that talks to the files and knows how to read bytes and interptret them as the variables we see in our analysis. Most of these drivers were develop to get the data from files on disk.


In [ ]:
%%time

downloaded_files = earthaccess.download(results, "./data/merra2")
downloaded_files

In [ ]:
!du -hs ./data

> POSIX paths are the Unix-style convention for representing file locations in local disk using / as the directory separator, such as /data/file.nc

### Now we are going to use xarray to compute a mean over a subset of the data, approx Colorado

* Our data is on disk, reading from local files is fast
* We are moving data from our disk to memory

In [ ]:
%%time

ds_from_disk = xr.open_mfdataset(downloaded_files)
ds_from_disk

### Computing on a locally loadad dataset

When we already load a local file as an Xarray dataset, our data reads are local and they are fast! 

In [ ]:
%%time

# Seems complicated but is not, just xarray notation to subset and calculate a mean for the 2 selected areas
t2m_mean = (
    ds_from_disk["T2M"]
    .sel(
        lat=slice(37, 41), lon=slice(-110, -102)
    )  # <- subset loads less data into memory!
    .mean(dim=["time"])
    .compute()
)
t2m_mean.plot()

Notice the sequencing here: **all** the bytes for both granules landed on disk *before* `xarray` ever opened a variable. If all we wanted was the global-mean 2-m temperature time series, we still wait for the full download.


## 5. Example B — the *"streaming"* workflow

Now let's do the equivalent thing without ever writing a file to disk. `earthaccess.open()` returns a list of file-like objects; `xarray` (via `h5netcdf`/`netCDF4`) reads from them lazily.


In [ ]:
%%time

file_objects = earthaccess.open(results)
file_objects

That call was fast — much faster than the download above — because it hasn't actually fetched the science data yet. It's only opened authenticated handles and read the minimal metadata `xarray` needs (dimensions, variable shapes, attributes) to build the dataset structure.


The next code block is the moment actual science-data bytes move over the network:
> **Note that **only** the T2M variable, specifically for the time/region requested, gets fetched.**

### This is called **Lazy-loading** or **data on-demand**

In [ ]:
%%time

ds_streamed = xr.open_mfdataset(file_objects)
# Seems complicated but is not, just xarray notation to subset and calculate a mean for the 2 selected areas
ds_streamed

Compare that to the download path: with `xr.open_mfdataset` over local files, the read is fast because the bytes are already on disk, but we paid the download cost earlier, for the *entire* file, whether or not we needed all of it. With streaming, the cost is paid incrementally, only where we touch the data. If all we needed was `T2M`, we never move the bytes for the other ~30 variables in the file at all.


### Computing on a remote dataset

When we stream data, we don't really get most of it until we compute something on it, this is called lazy-compute or lazy-loading, our data reads are slower than reading to local disk but we only read what we need! 

In [ ]:
%%time

t2m_mean = (
    ds_streamed["T2M"]
    .sel(
        lat=slice(37, 41), lon=slice(-110, -102)
    )  # <- subset is less data requested and less data loaded.
    .mean(dim=["time"])
    .compute()
)

t2m_mean.plot()

## 6. When to download vs. when to stream

There's no universally right answer, it depends on where our compute is and how we're going to use the data. The main factors to weigh are: **network cost/proximity** (in-region AWS reads are essentially free and fast; out-of-region reads pay real latency and sometimes egress cost per request), **access pattern** (one-shot full-file reuse favors downloading; sparse/subsetted or exploratory access favors streaming), **format support** (some formats can't be streamed at all — see Section T2 below), and **scale** (streaming avoids the storage-management burden of downloading hundreds or thousands of granules, but many small streamed reads can add up in request overhead if we're not careful about chunking/block size).

### The "S3 is our hard drive" mental model

When we're running inside AWS (particularly in `us-west-2`, where most Earthdata Cloud archives live), S3 access is essentially free and extremely fast — we're on the same network fabric as the data. In that setting it's useful to stop thinking of S3 as "a remote place we fetch things from" and start thinking of it as **our hard drive** — because we're renting compute next to a data lake that's already sized for the whole archive, and the object store *is* effectively our local disk. Downloading a copy onto our EC2 instance's own volume in that scenario mostly just costs us time and disk space for no benefit.

A good rule of thumb: **streaming trades network-request overhead for storage and full-file transfer cost.** Whether that's a good trade depends on how close (both physically and in cost) our compute is to the data, and how much of each file we actually plan to use.


**Exercise**: Complete the time to sience times for 2 MERRA files

**When we downloaded**:

* Disk: ~768MB 
* Time to sience (out of region) (good internet): ~1m
* TTS in-region(): ~ < 1m

**When we streamed**:

* Disk: 0
* Time to science (out of region): ~X seconds
* TTS in-region(): ~ <X seconds


## 7. "Cloud-conomics"


### **Downloading still makes sense when**:
- **We're outside AWS** (a laptop, an on-prem HPC cluster, a non-AWS cloud) — repeated small reads over the public internet, each with its own auth/TLS overhead, can end up slower *and* costlier in aggregate than one bulk transfer.
- **We need whole files repeatedly and we have enough disk space** — e.g., feeding it into a tool that only accepts local file paths, or reprocessing the same granules many times; paying the transfer cost once up front amortizes better. We have to have enough space in disk!
- **We need guaranteed, repeatable local access** — offline work, archival/reproducibility requirements, or environments with unreliable network access.
- **The reader library doesn't support remote/streamed I/O well** for our format, and a local copy sidesteps compatibility issues.



<img width="1440" height="568" alt="streaming-out" src="https://gist.github.com/user-attachments/assets/27de0678-1fbc-41fc-b6ea-61055e97459e" />

### **Streaming makes the most sense when**:

- **We're running in-region on AWS**, so egress is free/fast and streaming reads are cheap.
- **We only need a subset** of each file — a few variables, a spatial subset, a handful of time steps — out of a much larger granule.
- **We're doing exploratory analysis** and don't yet know which files/variables we'll actually need; streaming lets us look before committing to a download.
- **We don't want to manage storage** — no local disk cleanup, no worrying about running out of space when scaling across hundreds or thousands of granules.
- **We're building a virtual/aggregate dataset** (e.g., with `VirtualiZarr` or `kerchunk`) that references many source granules by URL — we want byte-range reads against the originals, not full local copies.

<img width="1440" height="568" alt="streaming-region" src="https://gist.github.com/user-attachments/assets/eb9ce7e5-9def-44bf-8ac2-68c9e4b2a1dc" />

## 8. Key takeaways


1. `earthaccess.login()` is required up front — for search, download, *and* streaming — because streaming means live, authenticated reads against Earthdata Cloud as we go.
2. **Downloading** moves an entire file's bytes to local disk before we can touch anything. **Streaming** (`earthaccess.open()`) returns lazy, `fsspec`-backed handles that only move the bytes our analysis actually accesses.
3. Choosing between them is about *where our compute lives relative to the data*: inside AWS (especially `us-west-2`), think of S3 as our hard drive and stream freely; outside AWS, or when we need the whole file repeatedly, downloading is usually more efficient.
4. Under the hood, `earthaccess`/`fsspec` don't issue a network request per byte-read — they fetch and cache data in **4 MB blocks**, which is what makes streaming practical instead of prohibitively chatty over the network.

### **Additional resources:**

- `earthaccess` docs: https://earthaccess.readthedocs.io/
- `earthaccess` GitHub: https://github.com/earthaccess-dev/earthaccess
- MERRA-2 documentation (GES DISC): https://disc.gsfc.nasa.gov/datasets?keywords=MERRA-2
- `fsspec` caching internals: https://filesystem-spec.readthedocs.io/en/latest/features.html#caching-files
- NASA Openscapes cloud-computing tutorials: https://nasa-openscapes.github.io/
- [NASA Glossary](../../glossary/nasa-glossary.md)
- [Cloud Computing GLossary](../../glossary/cloud-glossary.md)

## Technical Appendix 
    
## T1. Under the hood: how earthaccess actually fetches bytes
---

When we call `earthaccess.open()`, we get back `fsspec`-based file-like objects (backed by `s3fs` for direct S3 access, or `fsspec`'s HTTPS file system when going over HTTPS). These objects implement Python's file interface (`read()`, `seek()`, etc.) but instead of reading from local disk, each `read`/`seek` triggers a **ranged HTTP/S3 GET request** for just the bytes needed.

Naively, that could mean a *huge* number of tiny network requests — HDF5/NetCDF4 readers issue many small, scattered reads while parsing chunked/compressed variables, and turning each one into its own network round trip would be painfully slow. `fsspec` solves this with **block-based caching (`AbstractBufferedFile`)**: rather than fetching exactly the bytes requested, it fetches in fixed-size blocks — **`earthaccess` configures this at 4 MB by default** — and caches each block in memory. 

An important feature is that earthaccess chooses block_size based on file size (4/8/16 MB) via `_optimal_fsspec_block_size()` and passes it to `fsspec.open()` this way block size is adaptive but can be overriden via `open_kwargs`. See the way this works with the NISAR mission: **<a href="https://nisar-docs.asf.alaska.edu/earthaccess/">Using NISAR data with earthaccess</a>.**

- A small read for, say, 200 bytes of header still triggers a fetch of a full 4 MB block containing those bytes.
- Any *subsequent* read that falls within that already-cached block is served from memory — **no network request at all**.
- Reads are effectively pre-fetched in bulk, which amortizes request/connection overhead (and any per-request latency, like S3's or EDL's auth roundtrip) over many logical reads.

This is exactly the same idea as page caching in an OS, or read-ahead buffering on a local disk — just implemented over HTTP range requests instead of a block device. The 4 MB size is a deliberate tradeoff: large enough to make each network request worthwhile and amortize latency, small enough not to waste bandwidth pulling data we'll never touch when we only need a small slice of a big file.

#### Let's see it in practice by peeking at the underlying `fsspec` object.

In [ ]:
f = file_objects[0]
print(type(f))

# The cache object is where the block-fetching logic lives
print(type(f.cache))
print("Configured block size (bytes):", f.blocksize)
print("Configured block size (MB):", f.blocksize / (1024**2))

In [ ]:
# A first read anywhere in the file pulls (at least) one full block into the cache...
f.seek(0)
_ = f.read(10)
print(
    "Bytes cached after first tiny read:",
    f.cache.size if hasattr(f.cache, "size") else "n/a",
)

# ...and a subsequent nearby read is served from that cached block, with no new network call.
f.seek(100)
_ = f.read(10)

This block-caching behavior is *why* streaming can be competitive with — or faster than — downloading when we only need part of a file: we're not paying for a full sequential transfer, but we're also not paying a network round trip for every single small read either. The 4 MB block size is a sensible default for typical Earth science granule sizes and access patterns, but it's worth knowing it exists if we ever see unexpectedly heavy data transfer for what feels like a "small" read — e.g., touching one value from every chunk of a variable can still pull in a lot of 4 MB blocks.



## T2. Streamable Data Formats
---

Reading remote geospatial data efficiently comes down to one core difference: whether the format's underlying library demands an actual file on a hard drive, or if it can accept a flexible data stream. Formats like Zarr, HDF5, and NetCDF4 (using the h5netcdf engine) are flexible. They allow tools like fsspec to read just the exact bytes we need directly from cloud storage. Other formats achieve similar remote access through pure Python readers (NetCDF3), server-side queries (OPeNDAP), or GDAL's virtual file system (VSI) for Cloud Optimized GeoTIFFs.

On the flip side, older formats like HDF4, GRIB, and compressed Zip archives are rigid. Because they require a real, local file path to function, we are forced to download or cache the data before we can work with it. This technical divide is exactly why our ITS_LIVE pipeline succeeds with tools like VirtualiZarr and Kerchunk—they thrive on flexible, cloud-ready HDF5 data. If we were forced to incorporate rigid formats like HDF4, that entire remote-reading model would break down.

### Streamable

> **We can use file-like objects in Python**

| Format | Engine | How | Notes |
|---|---|---|---|
| **HDF5 / NetCDF4** | `h5netcdf` | `h5py.File(fileobj)` — h5py's core driver accepts any object with `read`/`seek`/`tell` | This is why fsspec/S3 remote reads work at all. Must use `engine="h5netcdf"`, not `engine="netcdf4"` (see below). |
| **Zarr** | `zarr` | fsspec `mapper` directly, no "file" concept at all — each chunk is its own object/key | Best-case format for cloud reads by design; this is why VirtualiZarr/Icechunk targets it. |
| **Kerchunk references** | `zarr` (via `fsspec.filesystem("reference")`) | Wraps byte-range references into a virtual Zarr store | Works *for HDF5-based sources*. Kerchunk has no real HDF4 backend, so this doesn't rescue HDF4. |
| **NetCDF3 classic / 64-bit offset** | `scipy` | `scipy.io.netcdf_file` is a pure-Python parser, reads from any file-like object | Older/simpler format; no compression, so this is genuinely lazy-capable. |
| **OPeNDAP** | `pydap` or `netcdf4` (via DAP URL) | Not "file-like" in the fsspec sense — it's a query protocol, server does the subsetting | Effectively remote-native; different mechanism than byte-range reads. |
| **COG / GeoTIFF** | `rasterio`/`rioxarray` | GDAL's own VSI virtual filesystem (`/vsicurl/`, `/vsis3/`, `/vsiaz/`) | Not a Python file-like object, but functionally equivalent — GDAL issues range requests itself. Works well because COGs are internally tiled/overviewed for exactly this. |
| **Parquet / GeoParquet** | *(not xarray directly, but PyArrow)* | PyArrow accepts fsspec file objects natively, lazy row-group/column reads | Relevant even though xarray doesn't read Parquet itself. |



### Not Streamable

> **The C library demands a real path (GDAL VSI, or Virtualization as the only workarounds)**

| Format | Engine | Why it fails | Workaround |
|---|---|---|---|
| **HDF4 / HDF-EOS2** | `pyhdf`/GDAL HDF4 driver | HDF4 C library has no virtual file driver interface at all | GDAL `/vsicurl/`/`/vsis3/` (partial — driver often reads most of the file anyway for chunked/compressed SDS), or fsspec cache-to-tempfile |
| **NetCDF4 via the `netCDF4` engine** | `netCDF4`/`netcdf4-python` | The underlying netCDF-C library only accepts a path, an OPeNDAP URL, or a fully-loaded in-memory byte blob (`diskless=True`) — not a streaming file-like object | Switch to `engine="h5netcdf"` for the same file (works if it's HDF5-based, which almost all modern NetCDF4 is) |
| **GRIB / GRIB2** | `cfgrib` | eccodes C library requires a real filesystem path; no file-like or VSI support | Download or cache locally first — there's no good lazy-remote path for GRIB today (unless virtualized) |
| **Zip archives (general)** | fsspec `zip://` | DEFLATE-compressed members can't be randomly seeked — decompression is inherently sequential from the start of the member | Only works losslessly if the zip was created with **no compression** (`ZIP_STORED`); then range reads into the member are possible since fsspec's zip filesystem can read the central directory (at the end) and then byte-range into an uncompressed member |
| **Any format accessed through `pyhdf`, `h5py` compiled without the `fileobj`/`ros3` driver, or older library builds** | varies | Depends on how the library was compiled — some HDF5 builds omit the S3 (`ros3`) driver even though the generic `fileobj` driver is usually present | Check with `h5py.get_config().ros3` or just test the fsspec-object approach, which doesn't need `ros3` |



The dividing line in one sentence: **if the format's reference C library was designed with a pluggable I/O layer (HDF5's VFD, Zarr's store abstraction), fsspec lazy reads work; if it assumes `open()`/`fread()` on a real file descriptor (HDF4, GRIB, netCDF-C's own NC_FORMAT path), we're stuck with GDAL VSI (where a driver exists) or local caching.**

> **Virtual Datasets**: In some cases, even if a file format is not streamable we can use the virtual data store paths (VDS), virtualization is a technique to create chunk references to the original files using the Zarr model, this allows us to lazy load data and bypass the native drivers for a given dataset. If we virtualize say GRIB or HDF5 files, we can open the virtual references using a Zarr driver. 



## T3. Streaming and downloading data in other languages
---

The same fundamental split — pluggable I/O layer vs. hard dependency on a real file path — shows up outside Python too:

| Language | Streaming tool | Notes |
|---|---|---|
| **R** | `stars`, `terra` (via GDAL VSI), `RNetCDF`/`ncdf4` (limited) | GDAL-backed R packages get `/vsicurl/`/`/vsis3/` streaming for free, same as `rioxarray` in Python. Native HDF5 reads via `RNetCDF` are more limited than `h5py`'s fileobj driver. |
| **Julia** | `Zarr.jl`, `NCDatasets.jl` (HDF5.jl backend) | `Zarr.jl` supports S3 stores natively; `HDF5.jl` streaming support depends on the same underlying HDF5 C library capabilities as Python's `h5py`. |
| **MATLAB** | Limited — mostly download-first | MATLAB's built-in `ncread`/`h5read` expect local paths or OPeNDAP URLs; no first-class fsspec-style byte-range streaming, so downloading (or using OPeNDAP) is the common path. |
| **Any language + GDAL bindings** | GDAL VSI (`/vsicurl/`, `/vsis3/`) | Since GDAL itself implements the virtual filesystem, any language with GDAL bindings (C++, Java, Node via `gdal-async`, etc.) inherits the same COG/VSI streaming behavior described in T2. |
| **Any language + OPeNDAP client** | DAP2/DAP4 protocol | OPeNDAP is language-agnostic by design — the server does the subsetting, so a thin client in any language can request just a slice without needing fsspec-style tooling at all. |

The takeaway: streaming capability tracks the *underlying C library* (HDF5, GDAL) far more than it tracks the language — `fsspec`/`h5py`/`s3fs` are Python's wrapper around the same HDF5 pluggable-I/O and GDAL VSI mechanisms other languages tap into directly.